🌐 [English](dx_app.ipynb) | **한국어** | [日本語](dx_app.ja.ipynb) | [中文](dx_app.zh.ipynb)

# DEEPX Tutorial 02 - DX-APP 사용법

이 두 번째 튜토리얼에서는 DX-APP을 소개하고, 컴파일된 DXNN 모델을 AI 애플리케이션에서 실행하는 방법을 배웁니다.

## 학습 목표

이 튜토리얼을 마치면 다음을 할 수 있습니다.

- DX-APP이 **[DEEPX Model Zoo](https://developer.deepx.ai/modelzoo/)의 모든 모델**(497개 모델 변형)에 대해 바로 실행할 수 있는 C++ 및 Python 예제 코드를 제공한다는 것을 알고, 그 코드로 원하는 모델을 실행할 수 있습니다.
- 예제가 작업(task), 모델 패밀리(family), 변형(variant)별로 어떻게 구성되어 있는지, 그리고 변형 이름이 모델 파일, 실행 파일, 스크립트를 어떻게 하나로 묶는지 설명할 수 있습니다.
- 튜토리얼에 필요한 모델과 영상만 다운로드할 수 있습니다.
- 분류, 검출, 포즈, 세그멘테이션 예제를 실행하고 저장된 결과를 노트북에서 확인할 수 있습니다.
- 동기, 비동기, PPU 실행 방식 중에서 선택하고, 어떤 Python 변형이 모델에 맞는지 측정할 수 있습니다.
- `dx_tool.sh`로 예제를 조회하고 데모 런처를 시작할 수 있습니다.
- 독립 실행형 C++ 또는 Python 예제를 내보내고, 그 모델 카드를 읽고, DX-APP 트리 밖에서 실행할 수 있습니다.
- 코드 대신 JSON 파일을 편집해 멀티 모델 그래프로 여러 모델을 하나의 파이프라인으로 연결할 수 있습니다.

## DX-APP이란?

**DX-APP**은 DEEPX NPU용 참조 애플리케이션입니다. 가장 큰 특징은 커버리지입니다. **[DEEPX Model Zoo](https://developer.deepx.ai/modelzoo/)에 공개된 모든 모델에는 바로 실행할 수 있는 예제**가 C++과 Python 양쪽으로 준비되어 있으며, 작업(object detection, classification, face detection, pose, segmentation, depth 등)별로 정리되어 있습니다. Model Zoo 페이지에서 모델을 고르면 DX-APP에 이미 그 모델을 실행하는 코드가 들어 있습니다. DX-APP은 개발자가 런타임 환경을 빠르게 구성하도록 돕고,
자신만의 AI 애플리케이션을 만들고 커스터마이즈하기 위한 템플릿 역할을 합니다.

이 튜토리얼은 DX-All Suite v2.5.0에 포함된 버전인 **DX-APP v3.3.0**을 사용합니다.

자세한 내용은 [dx_app git 저장소](https://github.com/DEEPX-AI/dx_app/blob/main/README.md)의 개요를 참고하거나, [여기](https://developer.deepx.ai/download/?id=744)에서 DX-APP User Guide를 다운로드하세요!

> **참고:** User Guide를 다운로드하려면 먼저 https://developer.deepx.ai/ 에 로그인해야 합니다.

DX-APP의 파일 구조를 살펴보겠습니다.

In [ ]:
# Load the shared DEEPX tutorial paths (identical first code cell in every notebook).
from pathlib import Path; import importlib, sys
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tutorial_paths.py").is_file())))
import tutorial_paths; importlib.reload(tutorial_paths)  # pick up updates without restarting the kernel
from tutorial_paths import setup_tutorial
import shlex

paths = setup_tutorial(needs=["dx_rt", "dx_app", "npu"])
paths.require(*["dx_rt", "dx_app", "npu"])   # stop here with the fix if a prerequisite is missing
%cd {DX_ALL_SUITE_DIR}


In [ ]:
!tree -L 1 dx-runtime

In [ ]:
!tree -L 1 dx-runtime/dx_app
!tree -L 1 dx-runtime/dx_app/bin
#!tree -L 2 dx-runtime/dx_app/src

**DX-APP** 데모는 최소한의 설정으로 DEEPX NPU에서 사전 컴파일된 모델을 시연하도록 최적화되어 있습니다.
각 데모는 일반적인 AI 작업을 나타내며 이미지, 영상, 실시간 카메라
입력으로 실행할 수 있습니다.

<img src="assets/ai-model-type.png" style="max-width: 1000px;">

---
**DX-APP 저장소 구성:**
```shell
dx_app/
├── src/
│   ├── cpp_example/                    # C++ end-to-end examples (497 model variants across 29 tasks)
│   │   ├── <task>/<family>/<variant>/  #   One folder per .dxnn file: entry points, factory, config.json, README.md
│   │   ├── multi_model_graph/          #   Multi-model graph CLI and sample graph JSON files (section 4)
│   │   └── common/                     # Shared C++ runtime layer
│   │       ├── base/                   #   Abstract interfaces (factories, pre/post-processors, visualizers, inputs)
│   │       ├── processors/             #   Shared processors
│   │       ├── runner/                 #   Task-specific sync/async runner pairs
│   │       ├── inputs/                 #   Image, video, camera, and RTSP input sources
│   │       ├── visualizers/            #   Task-specific visualizers
│   │       ├── config/                 #   ModelConfig loader
│   │       ├── graph/                  #   Multi-model graph engine
│   │       ├── registry/               #   Model registry used by the graph engine
│   │       ├── trackers/               #   IoU tracker
│   │       ├── third_party/            #   Header-only third-party libraries
│   │       └── utility/                #   Labels, preprocessing, profiling, and run utilities
│   ├── python_example/                 # Python end-to-end examples (the same 497 variants and 29 tasks)
│   │   ├── <task>/<family>/<variant>/  #   Four scripts per variant, config.json, and the model card files
│   │   └── common/                     # Shared Python runtime layer (same structure as the C++ one)
│   ├── postprocess/                    # C++ post-processing (consumed by pybind11 bindings)
│   ├── utility/                        # Shared support code used by build flow
│   └── bindings/
│       └── python/
│           ├── dx_postprocess/         # pybind11 bindings wrapping src/postprocess/
│           └── dx_graph/               # pybind11 module running multi-model graphs (./build.sh --all)
├── config/
│   ├── model_registry.json             # Model registry: one row per variant, the single source of truth
│   ├── test_models.conf                # Test model configuration
│   └── README.md                       # Config directory documentation
├── scripts/                            # dx_tool.sh, extract_model_package.sh, download_models.py, and helpers
├── tests/                              # pytest-based test suites
├── assets/                             # Links to downloaded models and videos
├── sample/                             # Bundled images and task-specific sample data
├── bin/                                # C++ executables generated by the build: <variant>_sync, <variant>_async
├── build.sh                            # Build script (default: the 26 demo models; --all: every variant)
├── setup.sh                            # Model, video, and dependency setup
├── run_demo.sh                         # Demo launcher: 26 demos, up to 6 execution modes
├── run_multi_model_demo.sh             # Multi-model graph scenarios and graph checks
├── install.sh                          # Dependency and OpenCV installer
└── docs/                               # Detailed documentation
```
---
**작업, 패밀리, 변형**

모든 예제는 `<task>/<family>/<variant>/`에 있습니다. **변형(variant)**은 `.dxnn` 파일 이름에서 확장자를 뺀 부분이며, 그 모델에 속한 모든 것의 이름이 됩니다.

| 항목 | `yolov7_640x640.dxnn`의 경우 |
|---|---|
| 예제 폴더 | `src/cpp_example/object_detection/yolov7/yolov7_640x640/` 및 `src/python_example/` 아래의 같은 경로 |
| C++ 실행 파일 | `bin/yolov7_640x640_sync`, `bin/yolov7_640x640_async` |
| Python 스크립트 | `yolov7_640x640_sync.py`, `yolov7_640x640_async.py`, `yolov7_640x640_sync_cpp_postprocess.py`, `yolov7_640x640_async_cpp_postprocess.py` |
| 다운로드 이름 | `./setup.sh --models yolov7_640x640` |

> **DX-APP v3.2.x에서 넘어오셨나요?** 이제 실행 파일은 변형 이름을 따릅니다(`bin/yolov7_async`는 이제 `bin/yolov7_640x640_async`). PPU 모델은 `object_detection/yolo_ppu/`로 옮겨졌고, 여러 작업의 이름이 바뀌거나 나뉘었습니다. `classification`은 이제 `image_classification`, `obb_detection`은 `oriented_object_detection`, `image_enhancement`는 `low_light_enhancement`, `face_alignment`는 `face_landmark`이며, `embedding`은 `face_recognition`, `image_retrieval`, `visual_place_recognition`, `zero_shot_image_classification`으로 나뉘었습니다.

**29개 작업 디렉터리** (`cpp_example`과 `python_example`에서 동일):

- 검출: `object_detection`, `oriented_object_detection`, `face_detection`, `hand_detection`, `mainbody_detection`, `3d_object_detection`, `anomaly_detection`
- 세그멘테이션: `instance_segmentation`, `semantic_segmentation`, `zero_shot_instance_segmentation`, `panoptic_driving_perception`, `image_matting`
- 포즈와 키포인트: `pose_estimation`, `hand_landmark`, `face_landmark`, `keypoint_detection`, `object_pose_estimation`
- 분류와 속성: `image_classification`, `zero_shot_image_classification`, `person_attribute`, `face_attribute`
- 인식과 검색: `face_recognition`, `person_reid`, `image_retrieval`, `visual_place_recognition`
- 이미지 복원과 깊이: `image_denoising`, `super_resolution`, `low_light_enhancement`, `depth_estimation`

DX-APP v3.3.0에는 Model Zoo 모델 147개가 추가로 등록되었습니다. RT-DETR(v1, v2, v3), YOLOv12, YOLOv13, 사전 최적화된 YOLO26, 그리고 새 작업용 모델인 이상 탐지(PatchCore, EfficientAD), 이미지 매팅(PP-Matting), CLIP 이미지 검색, 사람 재식별(re-identification), 시각적 장소 인식(EigenPlaces)이 포함됩니다.

## 사전 요구 사항

- 튜토리얼 01 완료: DX-Runtime을 `--all`로 설치(DX-RT, DX-APP 및 그 `venv-dx-runtime` Python 환경)하고 NPU가 `/dev/dxrt0`으로 인식되는 상태.
- **DX-RT 3.5.0 이상.** DX-APP v3.3.0은 Model Zoo 릴리스 `2_5_0`에서 모델을 다운로드하며, 이 릴리스의 `.dxnn` 파일은 컨테이너 형식 v9를 사용합니다. 이전 런타임은 이 파일을 거부합니다(문제 해결 참고). `dxrt-cli --version`으로 설치된 버전을 확인할 수 있습니다.
- DX-APP 샘플 모델과 영상. 튜토리얼 01의 3.3절에서 다운로드하며, 아래 다운로드 셀은 아직 없는 것만 받아옵니다(이 튜토리얼용 모델 약 300 MB, 새 호스트라면 공유 샘플 영상 아카이브 1.2 GB 추가).
- 그래픽 데스크톱 세션은 선택 사항입니다. 셀은 예제를 헤드리스(`--no-display --save`)로 실행하고 저장된 이미지를 노트북에 표시합니다. 창을 여는 셀은 `SHOW_WINDOW = False` 같은 플래그로 꺼져 있습니다.
- `sudo` 불필요. 소요 시간은 약 45분입니다. 두 셀은 각각 1~2분이 걸립니다. 3.3절의 C++ 빌드와, 그래프 실행 파일이 없는 경우 4.1절의 빌드입니다.

다음 셀은 SDK를 찾고, 이 요구 사항을 확인한 뒤 상태 표를 출력합니다. `MISSING`으로 표시된 항목에는 그것을 제공하는 단계가 함께 표시됩니다.

## 1. 모델과 영상 준비

1. `dx_app` 디렉터리로 이동합니다.

In [ ]:
# Move to the configured DX-APP directory.
%cd {DX_APP_DIR}

2. `setup.sh`로 필요한 모델과 샘플 영상을 다운로드합니다. 이 스크립트는 대화형(어떤 모델 카테고리를 다운로드할지 묻습니다)이므로 셀이 아니라 터미널에서 실행합니다. **File > New > Terminal**을 열고 다음을 실행하세요.

```bash
cd <DX_ALL_SUITE_DIR>/dx-runtime/dx_app
./setup.sh --no-force
```

3. 다음 셀은 실제 설치 경로를 넣어 같은 두 명령을 출력합니다.

In [ ]:
# Interactive script: run these two commands in a terminal (File > New > Terminal).
print(f"cd {shlex.quote(str(DX_APP_DIR))}")
print("./setup.sh --no-force")


> ![](assets/sc-setup.png)

4. 전체 Model Zoo를 받을 여유 저장 공간(수십 GB)이 있다면 **Enter**를 눌러 모든 Model Zoo 모델을 다운로드합니다. 스크린샷은 이전 릴리스의 것이며, 현재 메뉴에는 더 많은 모델과 새 작업 이름이 나열됩니다.

5. 그렇지 않다면 아래 코드 셀로 이 튜토리얼에 필요한 모델만 다운로드하세요. `yolov7_640x640`처럼 `.dxnn` 파일 이름에서 확장자를 뺀 **변형 이름**을 전달합니다. `YoloV7` 같은 이전 Model Zoo 이름도 여전히 사용할 수 있습니다. 튜토리얼 01의 3.3절에서 이 모델들을 이미 다운로드했으므로, 그 절을 완료했다면 `--no-force` 덕분에 셀이 모든 파일을 건너뛰고 몇 초 만에 끝납니다. 3절과 4절도 같은 방식으로 추가 모델을 받아옵니다.

In [ ]:
# The models used in section 2, by variant name (the .dxnn file stem).
# --no-force skips files that already exist, so rerunning this cell is cheap.
SELECTED_MODELS = shlex.join([
    "resnet50_224x224",
    "yolov7_640x640",
    "yolov7_640x640_ppu",
    "yolov8-s-pose_640x640",
    "yolov8-n-seg_640x640",
])

# Preview what would be downloaded, without downloading:
#!cd "{DX_APP_DIR}" && bash setup.sh --models {SELECTED_MODELS} --dry-run

# Download the selected models only when they are missing
!cd "{DX_APP_DIR}" && bash setup.sh --models {SELECTED_MODELS} --no-force

6. 모델과 영상이 모두 정상적으로 다운로드되었는지 확인합니다.

In [ ]:
# Downloaded models and videos are stored under `assets` path as a symbolic link
!cd "{DX_APP_DIR}" && tree assets

In [ ]:
# AI models converted to DXNN format

# Number of downloaded models
!cd "{DX_APP_DIR}" && ls assets/models | wc -l

# Downloaded model list
!cd "{DX_APP_DIR}" && tree assets/models

In [ ]:
# video files for demo inputs
!cd "{DX_APP_DIR}" && tree assets/videos

## 2. 데모 실행

**DX-APP**은 분류, 검출, 세그멘테이션, 포즈 추정 등 컴파일된 모델을 DEEPX NPU에서 실행하는 방법을 보여 주는 바로 사용할 수 있는 데모 애플리케이션 모음입니다.

사용자는 다양한 DX-APP 데모를 살펴보며 애플리케이션을 배우고 만들 수 있습니다.

`bin/`의 C++ 실행 파일은 변형 이름을 따라 `bin/<variant>_sync`와 `bin/<variant>_async`로 이름이 붙습니다. 모두 같은 입력 옵션(`-i` 이미지, `-v` 영상, `-c` 카메라, `-r` RTSP 스트림)과 같은 출력 옵션을 받습니다.

| 옵션 | 효과 |
|---|---|
| `--no-display` | 창을 열지 않고 성능 요약만 출력 |
| `-s`, `--save` | 렌더링된 결과를 저장 |
| `--save-dir <dir>` | 저장 위치. 실행할 때마다 새 하위 디렉터리를 만듦 |
| `-l`, `--loop <n>` | 입력을 `n`번 반복 |

이 절의 셀은 `--no-display --save`로 헤드리스 실행하고 저장된 이미지를 셀 아래에 표시하므로, SSH 환경에서도 동작하며 노트북을 멈추게 하지 않습니다. 주석 처리된 줄은 같은 예제를 영상이나 카메라로 실행합니다. 이 줄은 창을 열기 때문에 데스크톱 세션에서 사용하세요.

다음 셀은 출력 디렉터리와, 가장 최근에 저장된 이미지를 보여 주는 작은 헬퍼를 만듭니다.

In [ ]:
import time
from IPython.display import Image, Markdown, display

DEMO_OUTPUT_DIR = WORKSPACE_DIR / "dx-app-outputs"   # results saved with --save in section 2
DEMO_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Remember when each cell starts, so the helper shows only images saved by the current cell.
CELL_START = {"time": time.time()}
def _mark_cell_start(info):
    CELL_START["time"] = time.time()
ip = get_ipython()
for callback in list(ip.events.callbacks["pre_run_cell"]):
    if getattr(callback, "__name__", "") == "_mark_cell_start":
        ip.events.unregister("pre_run_cell", callback)
ip.events.register("pre_run_cell", _mark_cell_start)


def show_latest_image(folder, width=640):
    # Show the newest .jpg/.png that the current cell saved under `folder`
    # (every run saves into its own sub-directory).
    images = [p for p in Path(folder).rglob("*")
              if p.suffix.lower() in (".jpg", ".jpeg", ".png") and p.stat().st_mtime >= CELL_START["time"]]
    if not images:
        print(f"This cell saved no image under {folder}; check the command output above.")
        return
    latest = max(images, key=lambda p: p.stat().st_mtime)
    print("Saved result:", latest.relative_to(folder))
    display(Image(filename=str(latest), width=width))

### 2.1 분류
- **bin/resnet50_224x224_sync**는 사전 학습된 ImageNet 분류 모델을 실행해 가장 가능성이 높은 클래스를 이미지 위에 그립니다.

`-m`은 모든 C++ 예제에서 선택 사항입니다. 지정하지 않으면 예제는 기본 모델 `assets/models/<variant>.dxnn`을 불러옵니다. 그래도 셀에서는 모델이 어디에서 오는지 보이도록 이 옵션을 전달합니다.

In [ ]:
%cd {DX_APP_DIR}

# Help: the options shared by every C++ example
!./bin/resnet50_224x224_sync -h

# Image input, headless: classify one ImageNet sample and save the annotated image
!./bin/resnet50_224x224_sync -m assets/models/resnet50_224x224.dxnn -i sample/ILSVRC2012/1.jpeg --no-display --save --save-dir "{DEMO_OUTPUT_DIR}"
show_latest_image(DEMO_OUTPUT_DIR)

### 2.2 객체 검출
- **bin/yolov7_640x640_sync**와 **bin/yolov7_640x640_async**는 YOLOv7 객체 검출 모델을 이미지, 영상 또는 카메라에 대해 실행합니다. 다음 셀은 기본 빌드가 만든 다른 YOLO 실행 파일 목록을 보여 줍니다.

In [ ]:
%cd {DX_APP_DIR}

# Find other YOLO-based reference executables
!ls ./bin/yolo*

#### 실행 방식

DX-APP은 다양한 애플리케이션 요구 사항에 맞는 여러 실행 방식을 제공합니다.

| 방식 | 주요 특징 | 권장 용도 |
|---|---|---|
| 동기(Synchronous) | 각 프레임을 순차적으로 처리 | 단순한 흐름과 지연 시간 확인 |
| 비동기(Asynchronous) | 파이프라인 단계를 겹쳐서 실행 | 더 높은 처리량 |
| PPU | 지원되는 후처리 작업을 오프로드 | 호스트 CPU 부하 감소 |

- **동기 애플리케이션**

<img src="https://github.com/DEEPX-AI/dx_rt/raw/main/docs/source/resources/09_01_Sync_Inference_Operation.png" style="max-width: 300px;">

In [ ]:
%cd {DX_APP_DIR}

# Image input, headless: the annotated image is saved and shown below
!./bin/yolov7_640x640_sync -m assets/models/yolov7_640x640.dxnn -i sample/img/sample_dog.jpg --no-display --save --save-dir "{DEMO_OUTPUT_DIR}"
show_latest_image(DEMO_OUTPUT_DIR)

# Video input, headless: 588 frames, prints the performance summary (about 20 seconds)
!./bin/yolov7_640x640_sync -m assets/models/yolov7_640x640.dxnn -v assets/videos/boat.mp4 --no-display

# On a desktop session (opens a window):
#!./bin/yolov7_640x640_sync -m assets/models/yolov7_640x640.dxnn -v assets/videos/boat.mp4
#!./bin/yolov7_640x640_sync -m assets/models/yolov7_640x640.dxnn -c 0

- **비동기 애플리케이션**


<img src="https://github.com/DEEPX-AI/dx_rt/raw/main/docs/source/resources/09_01_Async_Inference_Operation.png" style="max-width: 1000px;">

다음 셀은 같은 영상을 비동기 실행 파일로 실행합니다. 출력되는 `Overall FPS` 줄을 위의 동기 실행 결과와 비교해 보세요.

In [ ]:
%cd {DX_APP_DIR}

# Video input, headless: compare "Overall FPS" with the synchronous run above
!./bin/yolov7_640x640_async -m assets/models/yolov7_640x640.dxnn -v assets/videos/boat.mp4 --no-display

# On a desktop session (opens a window):
#!./bin/yolov7_640x640_async -m assets/models/yolov7_640x640.dxnn -v assets/videos/boat.mp4
#!./bin/yolov7_640x640_async -m assets/models/yolov7_640x640.dxnn -c 0

# RTSP input (this public stream is reachable only from Korea)
#!./bin/yolov7_640x640_async -m assets/models/yolov7_640x640.dxnn -r "rtsp://210.99.70.120:1935/live/cctv002.stream"

- **PPU와 비PPU 비교**

아래 두 셀은 같은 7초짜리 영상을 비PPU 모델(`bin/yolov7_640x640_async`)과 PPU 모델(`bin/yolov7_640x640_ppu_async`)로 각각 실행합니다. 기본값인 `SHOW_WINDOW = False`에서는 `--no-display`로 헤드리스 실행되어 FPS 요약만 출력하므로 노트북이 멈추지 않습니다. 데스크톱 세션에서 `SHOW_WINDOW = True`로 설정하면 DX-APP 창에서 검출 결과와 FPS 카운터를 볼 수 있습니다. 두 FPS 줄을 비교해 보세요. PPU 모델은 후보 필터링과 클래스 선택을 NPU로 옮깁니다. 모든 PPU 예제는 `object_detection/yolo_ppu/` 아래에 있습니다.

In [ ]:
SHOW_WINDOW = False   # True opens the DX-APP window (desktop session only); False prints the FPS summary and returns
%cd {DX_APP_DIR}

# PPU disabled
!./bin/yolov7_640x640_async -m assets/models/yolov7_640x640.dxnn -v assets/videos/drone-city-road2.mov {"" if SHOW_WINDOW else "--no-display"}

In [ ]:
SHOW_WINDOW = False   # keep the same value as in the previous cell
%cd {DX_APP_DIR}

# PPU enabled
!./bin/yolov7_640x640_ppu_async -m assets/models/yolov7_640x640_ppu.dxnn -v assets/videos/drone-city-road2.mov {"" if SHOW_WINDOW else "--no-display"}

### 2.3 포즈 추정
- **bin/yolov8-s-pose_640x640_sync**는 포즈 추정 모델을 실행해 이미지, 영상 또는 카메라에서 사람을 검출하고 신체 키포인트를 추정합니다.

In [ ]:
%cd {DX_APP_DIR}

# Image input, headless
!./bin/yolov8-s-pose_640x640_sync -m assets/models/yolov8-s-pose_640x640.dxnn -i sample/img/sample_person_b.jpg --no-display --save --save-dir "{DEMO_OUTPUT_DIR}"
show_latest_image(DEMO_OUTPUT_DIR)

# On a desktop session (opens a window):
#!./bin/yolov8-s-pose_640x640_sync -m assets/models/yolov8-s-pose_640x640.dxnn -v assets/videos/dance-solo.mov
#!./bin/yolov8-s-pose_640x640_sync -m assets/models/yolov8-s-pose_640x640.dxnn -c 0

### 2.4 세그멘테이션
- **bin/yolov8-n-seg_640x640_async**는 인스턴스 세그멘테이션 모델을 실행해 이미지, 영상 또는 카메라에서 찾은 모든 객체에 대해 박스와 픽셀 마스크를 생성합니다.

In [ ]:
%cd {DX_APP_DIR}

# Image input, headless
!./bin/yolov8-n-seg_640x640_async -m assets/models/yolov8-n-seg_640x640.dxnn -i sample/img/sample_person_b.jpg --no-display --save --save-dir "{DEMO_OUTPUT_DIR}"
show_latest_image(DEMO_OUTPUT_DIR)

# On a desktop session (opens a window):
#!./bin/yolov8-n-seg_640x640_async -m assets/models/yolov8-n-seg_640x640.dxnn -v assets/videos/blackbox-city-road.mp4
#!./bin/yolov8-n-seg_640x640_async -m assets/models/yolov8-n-seg_640x640.dxnn -c 0

### 2.5 데모 런처: run_demo.sh

`run_demo.sh`는 작업마다 하나씩 **26개 데모**를 최대 **6가지 실행 모드**(C++ 동기와 비동기, 그리고 네 가지 Python 변형)로 실행합니다. 데모, 모드, 입력의 세 가지를 물어보므로, 터미널(**File > New > Terminal**)에서 시작한 뒤 메뉴에 답하세요.

```bash
cd <DX_ALL_SUITE_DIR>/dx-runtime/dx_app
./run_demo.sh
```

다음 셀은 실제 경로를 넣어 같은 명령을 출력합니다. 런처는 첫 메뉴를 표시하기 전에 26개 데모 모델 중 없는 것을 다운로드하므로, 처음 시작할 때는 몇 분이 걸릴 수 있습니다. 또한 `venv-dx-runtime` Python을 스스로 찾으므로, Python 모드를 위해 가상 환경을 활성화할 필요가 없습니다.

In [ ]:
# Interactive script: run these two commands in a terminal (File > New > Terminal).
print(f"cd {shlex.quote(str(DX_APP_DIR))}")
print("./run_demo.sh")


첫 번째 메뉴에는 데모 목록이 표시됩니다.

| # | 데모 | # | 데모 |
|---|---|---|---|
| 0 | Object Detection (YOLOv7) | 13 | Image Enhancement (Zero-DCE) |
| 1 | Object Detection (YOLOv11N) | 14 | Embedding (ArcFace) |
| 2 | Face Detection (SCRFD500M) | 15 | Attribute Recognition (DeepMAR-ResNet50) |
| 3 | OBB Detection (YOLO26N-OBB) | 16 | PPU Pipeline (YOLOv7-PPU) |
| 4 | Pose Estimation (YOLOv8s-Pose) | 17 | Keypoint Detection (SuperPoint) |
| 5 | Hand Landmark (HandLandmarkLite) | 18 | Object Pose Estimation (DOPE) |
| 6 | Face Alignment (3DDFA-V2-MobileNetV1) | 19 | Panoptic Driving (YOLOPv2) |
| 7 | Instance Segmentation (YOLOv8N-Seg) | 20 | 3D Object Detection (SFA3D) |
| 8 | Semantic Segmentation (DeepLabV3+MobileNet) | 21 | Hand Detection (MediaPipe Palm) |
| 9 | Classification (ResNet50) | 22 | Image Retrieval (CLIP RN50) |
| 10 | Depth Estimation (YOLO26-Depth-S) | 23 | Visual Place Recognition (EigenPlaces R18) |
| 11 | Image Denoising (DnCNN-50) | 24 | Person Re-ID (RepVGG-A0) |
| 12 | Super Resolution (ESPCN-X4) | 25 | Image Matting (PP-Matting HRNet-W48) |

두 번째 메뉴에서는 실행 모드를 고릅니다. 모드 1과 2는 C++ 실행 파일을, 모드 3~6은 3.4절에서 설명하는 네 가지 Python 변형을 실행합니다.

> ![](assets/sc-run_demo2.png)

세 번째 메뉴에서는 입력을 고릅니다. `1`은 데모의 샘플 영상, `2`는 샘플 이미지입니다.

세 가지 답은 옵션 `--task`, `--mode`, `--input`으로도 지정할 수 있으며, 이 경우 모든 메뉴를 건너뜁니다. 다음 셀은 런처의 도움말을 출력합니다. `RUN_DEMO_LAUNCHER = True`이면 이어서 데모 `0`(YOLOv7)을 모드 `2`(C++ 비동기)로 샘플 이미지에 대해 시작합니다. 데모는 창을 열기 때문에 이 플래그는 데스크톱 세션에서만 설정하세요.

In [ ]:
RUN_DEMO_LAUNCHER = False   # True starts the demo below; it opens a window (desktop session only)
%cd {DX_APP_DIR}

!./run_demo.sh --help

if RUN_DEMO_LAUNCHER:
    # Demo 0 (Object Detection, YOLOv7), mode 2 (cpp_async), input 2 (image): no menus
    !./run_demo.sh --task 0 --mode 2 --input 2

### 2.6 dx_tool.sh로 예제 찾기, 살펴보기, 내보내기

`scripts/dx_tool.sh`는 예제를 관리합니다. 인자 없이 시작하면 대화형 메뉴가 열리지만, 조회 명령은 바로 실행할 수도 있어 노트북에서 사용하기 편리합니다.

| 명령 | 하는 일 |
|---|---|
| `dx_tool.sh list [FILTER]` | 예제 목록 출력. 작업으로 필터링 가능(예: `det`) |
| `dx_tool.sh search <KEYWORD>` | 모델 이름이나 작업으로 예제 검색 |
| `dx_tool.sh info <MODEL>` | 한 예제의 파일, 팩토리 클래스, `config.json` 표시 |
| `dx_tool.sh run` / `dx_tool.sh bench` | 여러 예제를 한 번에 실행하거나 벤치마크(대화형 필터) |
| `dx_tool.sh extract` | 독립 실행형 패키지 내보내기(대화형) |

다음 셀은 YOLO26-S 예제를 검색하고 검출 모델의 세부 정보를 출력합니다. `search`는 각 결과를 `<task>/<family>/<variant>` 형식으로 출력하는데, 이는 3절의 내보내기 스크립트가 받는 형식과 정확히 같습니다. `info`는 기본 샘플 이미지와 영상, 그리고 `config.json`의 점수 및 NMS 임곗값 등을 보여 줍니다.

In [ ]:
%cd {DX_APP_DIR}

# Which examples exist for YOLO26-S? (C++ and Python, every task)
!./scripts/dx_tool.sh search yolo26-s

# Details of one variant: files, factory classes, and config.json
!./scripts/dx_tool.sh info yolo26-s_640x640

대화형 메뉴에서는 내보내기도 할 수 있습니다. 질문을 하므로 터미널(File > New > Terminal)에서 실행하세요.

```bash
cd <DX_ALL_SUITE_DIR>/dx-runtime/dx_app
./scripts/dx_tool.sh
```

다음 셀은 실제 경로를 넣어 같은 명령을 출력합니다. 메뉴가 나타나면 그 아래의 단계를 따르세요.

In [ ]:
# Interactive script: run these two commands in a terminal (File > New > Terminal).
print(f"cd {shlex.quote(str(DX_APP_DIR))}")
print("./scripts/dx_tool.sh")


메뉴가 나타나면:
- `2`(Extract Package)를 선택합니다.
- 프로그래밍 언어를 선택합니다. `1` C++, `2` Python, `3` 둘 다.
- 예제 경로를 `<task>/<family>/<variant>` 형식으로 입력합니다. 예: `object_detection/yolo26/yolo26-s_640x640`. Tab 키로 경로가 자동 완성됩니다.
- Enter를 눌러 기본 출력 디렉터리 `output`을 그대로 사용합니다.

> ![](assets/sc-extract.png)

스크린샷은 경로가 두 단계(`object_detection/yolo26s`)뿐이던 이전 릴리스의 것입니다. 몇 초 뒤 패키지가 `output/cpp/<task>/<family>/<variant>`와 `output/py/<task>/<family>/<variant>`에 생성됩니다.

이 메뉴는 내부적으로 `scripts/extract_model_package.sh`를 호출합니다. 3절에서는 이 스크립트를 직접 사용하는데, 모델과 언어를 인자로 받기 때문에 노트북 셀 안에서 실행할 수 있기 때문입니다.

## 3. 추출한 독립 실행형 예제 실행

2.6절에서는 대화형 내보내기를 살펴보았습니다. 이 절에서는 같은 작업을 노트북에서 `scripts/extract_model_package.sh`로 수행한 뒤, 내보낸 패키지를 **DX-APP 트리 밖에서** 빌드하고 실행합니다. 여러분의 프로젝트에서 하는 것과 똑같은 방식입니다.

| 절 | 예제(`<task>/<family>/<variant>`) | 언어 및 변형 | 보여 주는 내용 |
|---|---|---|---|
| 3.2 | `object_detection/yolo26/yolo26-s_640x640` | Python, `*_sync.py` | Model Zoo 모델을 실행하는 가장 간단한 방법 |
| 3.3 | `object_detection/yolo26/yolo26-s_640x640` | C++, `*_sync` | 내보낸 C++ 패키지를 CMake로 빌드 |
| 3.4 | `pose_estimation/yolo26_pose/yolo26-s-pose_640x640` | Python, `*_async_cpp_postprocess.py` | 다른 작업, C++ 후처리 사용 |
| 3.5 | `object_detection/yolo26/yolo26-s_640x640` | Python, 네 가지 변형 모두 | 사용할 변형을 측정으로 결정 |

내보낸 모든 Python 예제는 DX-Runtime이 `dx-runtime/venv-dx-runtime`에 설치한 `dx_engine`과 `dx_postprocess`를 임포트합니다. 따라서 셀은 그 인터프리터를 명시적으로 호출하고, Python 패키지는 `--no-prune`으로 내보냅니다(이유는 3.1절에서 설명합니다). DX-APP 트리 안에서는 입력을 주지 않으면 예제가 기본 샘플 이미지를 사용하지만, 내보낸 패키지에는 샘플 이미지가 없습니다. 따라서 셀은 `dx_app/sample/img`의 이미지를 전달해 헤드리스(`--no-display --save`)로 실행한 뒤 저장된 결과를 노트북에 표시합니다.

### 3.1 패키지 내보내기

먼저 스크립트 자체의 도움말부터 확인합니다. 다음 셀에서 사용하는 인자인 `<task>/<family>/<variant>` 형식의 예제(패밀리의 모든 변형이라면 `<task>/<family>`), `--lang cpp|py|both`, `--output-dir`, `--no-prune`이 나열됩니다.

In [ ]:
!cd "{DX_APP_DIR}" && bash scripts/extract_model_package.sh --help


`--output-dir` 없이 실행하면 스크립트가 `dx_app` 안에 패키지를 그대로 준비하고, 지정하면 패키지가 `<dir>/cpp/<task>/<family>/<variant>`와 `<dir>/py/<task>/<family>/<variant>`로 복사됩니다. 다음 셀은 YOLO26 모델 두 개를 다운로드하고, 세 패키지를 이 튜토리얼의 `workspace/` 디렉터리로 내보냅니다. 이 디렉터리는 git이 무시합니다. 세 패키지는 C++ 객체 검출 모델, 그리고 Python 객체 검출 모델과 포즈 추정 모델입니다.

In [ ]:
STANDALONE_DIR = WORKSPACE_DIR / "standalone"
OUTPUT_DIR = WORKSPACE_DIR / "standalone-outputs"
DX_RUNTIME_PYTHON = DX_RUNTIME_DIR / "venv-dx-runtime" / "bin" / "python"   # has dx_engine and dx_postprocess
SAMPLE_IMG_DIR = DX_APP_DIR / "sample" / "img"
MODEL_DIR = DX_WORKSPACE_DIR / "res" / "models"

for path in (STANDALONE_DIR, OUTPUT_DIR):
    path.mkdir(parents=True, exist_ok=True)
if not DX_RUNTIME_PYTHON.is_file():
    raise FileNotFoundError(f"DX-Runtime Python environment not found: {DX_RUNTIME_PYTHON} (Tutorial 01, section 3)")

# Both models were downloaded in Tutorial 01 section 3.3; this only downloads what is missing.
!cd "{DX_APP_DIR}" && bash setup.sh --models yolo26-s_640x640 yolo26-s-pose_640x640 --no-force

# Export: object detection in C++ (pruned, the default) and in Python, pose estimation in Python.
# The Python packages use --no-prune; the next markdown cell explains why.
!cd "{DX_APP_DIR}" && bash scripts/extract_model_package.sh object_detection/yolo26/yolo26-s_640x640 --lang cpp --output-dir "{STANDALONE_DIR}"
!cd "{DX_APP_DIR}" && bash scripts/extract_model_package.sh object_detection/yolo26/yolo26-s_640x640 --lang py --output-dir "{STANDALONE_DIR}" --no-prune
!cd "{DX_APP_DIR}" && bash scripts/extract_model_package.sh pose_estimation/yolo26_pose/yolo26-s-pose_640x640 --lang py --output-dir "{STANDALONE_DIR}" --no-prune

!tree -L 5 "{STANDALONE_DIR}"

#### 프루닝된 패키지와 `--no-prune`

기본적으로 내보내기는 `common/`을 **프루닝(pruning)**합니다. 즉 런타임 레이어 전체가 아니라 이 예제가 실제로 사용하는 공유 파일만 복사합니다. 패키지가 더 작고 읽기 쉬워지지만, 이 예제만 실행할 수 있습니다. `--no-prune`은 `common/` 전체를 복사하며, 같은 패키지에 두 번째 모델을 추가할 계획이라면 이 옵션도 필요합니다. 이전 셀의 C++ 패키지는 프루닝된 것입니다. 다음 셀은 Python YOLO26-S 예제를 기본 프루닝으로 한 번 더 내보내고 전체 복사본과 비교합니다.

> **DX-APP v3.3.0 관련 참고:** 프루닝된 **Python** 패키지에는 `config.json`에 지정된 프로세서 클래스가 빠져 있어, 시작 시 `VariantConfigError: processor class 'LetterboxPreprocessor' is exported by neither common.processors nor common.visualizers` 오류로 멈춥니다. 이 문제가 수정될 때까지는 이 튜토리얼처럼 Python 패키지를 `--no-prune`으로 내보내세요. 프루닝된 C++ 패키지는 영향을 받지 않습니다.

In [ ]:
PRUNED_DIR = WORKSPACE_DIR / "standalone-pruned"
!cd "{DX_APP_DIR}" && bash scripts/extract_model_package.sh object_detection/yolo26/yolo26-s_640x640 --lang py --output-dir "{PRUNED_DIR}"


def count_files(folder):
    return sum(1 for p in Path(folder).rglob("*") if p.is_file())


PACKAGE = Path("py") / "object_detection" / "yolo26" / "yolo26-s_640x640"
print(f"\n{'package':<14}{'all files':>10}{'common/ files':>15}")
for label, root in (("pruned", PRUNED_DIR), ("--no-prune", STANDALONE_DIR)):
    folder = root / PACKAGE
    print(f"{label:<14}{count_files(folder):>10}{count_files(folder / 'common'):>15}")

#### 모델 카드

모든 예제 폴더에는 Model Zoo에서 생성된 **모델 카드**가 들어 있습니다. `README.md`는 모델이 무엇인지, 출처와 함께 정확도와 속도, 다운로드, 실행 방법을 설명합니다. Python 예제 폴더에는 브라우저에서 오프라인으로 볼 수 있는 같은 카드인 `model.html`과, 참조 결과인 `result.webp`도 있습니다. 내보낸 Python 패키지는 이 세 파일을 모두 유지합니다. 다음 셀은 방금 내보낸 패키지의 참조 이미지와 카드를 보여 줍니다.

In [ ]:
import re

CARD_DIR = STANDALONE_DIR / "py" / "object_detection" / "yolo26" / "yolo26-s_640x640"
!ls "{CARD_DIR}"

display(Image(filename=str(CARD_DIR / "result.webp"), width=480))   # the reference result
card = (CARD_DIR / "README.md").read_text()
display(Markdown(re.sub(r"!\[[^\]]*\]\(result\.webp\)", "", card)))  # the card; its image is shown above
print("Offline HTML card:", CARD_DIR / "model.html")

### 3.2 예제 1: Python 독립 실행형, 객체 검출

`yolo26-s_640x640_sync.py`는 가장 기본적인 동기 변형으로, 순수 Python 후처리를 사용해 한 번에 한 프레임씩 처리합니다. 가장 먼저 읽어 볼 변형입니다. 인자는 DX-APP의 모든 Python 예제에서 동일합니다. `--model`, 입력 옵션 하나(`--image`, `--video`, `--camera`, `--rtsp`), 그리고 헤드리스 실행을 위한 `--no-display --save`입니다.

```bash
cd <workspace>/standalone/py/object_detection/yolo26/yolo26-s_640x640
<dx-runtime>/venv-dx-runtime/bin/python yolo26-s_640x640_sync.py \
    --model <workspace>/res/models/yolo26-s_640x640.dxnn \
    --image <dx_app>/sample/img/sample_street.jpg \
    --no-display --save --save-dir <output-dir>
```

In [ ]:
EXAMPLE_DIR = STANDALONE_DIR / "py" / "object_detection" / "yolo26" / "yolo26-s_640x640"
MODEL = MODEL_DIR / "yolo26-s_640x640.dxnn"
IMAGE = SAMPLE_IMG_DIR / "sample_street.jpg"

!cd "{EXAMPLE_DIR}" && "{DX_RUNTIME_PYTHON}" yolo26-s_640x640_sync.py --model "{MODEL}" --image "{IMAGE}" --no-display --save --save-dir "{OUTPUT_DIR}"

show_latest_image(OUTPUT_DIR)

### 3.3 예제 2: C++ 독립 실행형, 객체 검출

내보낸 C++ 패키지에는 `yolo26-s_640x640_sync.cpp`, `yolo26-s_640x640_async.cpp`, 공유 `common/` 레이어, 그리고 설치된 DX-RT에 링크하는 생성된 `CMakeLists.txt`가 들어 있습니다. 빌드는 표준 CMake 절차이며, DX-APP 트리를 참조하는 것은 아무것도 없습니다.

```bash
cd <workspace>/standalone/cpp/object_detection/yolo26/yolo26-s_640x640
mkdir -p build && cd build
cmake .. && make -j$(nproc)
./yolo26-s_640x640_sync -m <workspace>/res/models/yolo26-s_640x640.dxnn \
    -i <dx_app>/sample/img/sample_street.jpg --no-display --save --save-dir <output-dir>
```

C++ 예제는 옵션을 `-m/--model_path`와 `-i/--image_path`로 표기한다는 점에 유의하세요. `./yolo26-s_640x640_sync --help`로 목록을 볼 수 있습니다. 빌드에는 1~2분이 걸립니다.

In [ ]:
CPP_DIR = STANDALONE_DIR / "cpp" / "object_detection" / "yolo26" / "yolo26-s_640x640"

!cd "{CPP_DIR}" && mkdir -p build && cd build && cmake .. && make -j$(nproc)

!cd "{CPP_DIR}/build" && ./yolo26-s_640x640_sync -m "{MODEL}" -i "{IMAGE}" --no-display --save --save-dir "{OUTPUT_DIR}"

show_latest_image(OUTPUT_DIR)

### 3.4 예제 3: Python 독립 실행형, 포즈 추정, C++ 후처리를 사용하는 비동기

모든 Python 예제는 네 가지 변형으로 제공됩니다. 파일 이름을 보면 어떤 변형을 실행하는지 알 수 있습니다.

| 변형 | 후처리 | 스레딩 | 용도 |
|---|---|---|---|
| `*_sync.py` | 순수 Python | 동기 | 로직 학습과 디버깅 |
| `*_async.py` | 순수 Python | 비동기 | 더 높은 처리량. 배포를 위한 일반적인 출발점 |
| `*_sync_cpp_postprocess.py` | C++ 바인딩(`dx_postprocess`) | 동기 | 무거운 후처리를 한 번에 한 프레임씩 |
| `*_async_cpp_postprocess.py` | C++ 바인딩(`dx_postprocess`) | 비동기 | 높은 프레임 속도에서 무거운 후처리 |

> DX-APP v3.2.x에서는 `*_cpp_postprocess.py` 스크립트가 아무 표시 없이 Python 후처리기를 실행했으며, v3.3.0에서 이 문제가 수정되었습니다. 따라서 이전 릴리스로 측정한 결과에서는 변형 간 차이가 나타나지 않습니다.

이 예제는 다른 작업과 C++ 후처리를 사용하는 비동기 변형으로 전환합니다. 명령 형태는 바뀌지 않으며, 디렉터리, 스크립트 이름, 모델만 달라집니다.

```bash
cd <workspace>/standalone/py/pose_estimation/yolo26_pose/yolo26-s-pose_640x640
<dx-runtime>/venv-dx-runtime/bin/python yolo26-s-pose_640x640_async_cpp_postprocess.py \
    --model <workspace>/res/models/yolo26-s-pose_640x640.dxnn \
    --image <dx_app>/sample/img/sample_people.jpg \
    --no-display --save --save-dir <output-dir>
```

In [ ]:
POSE_DIR = STANDALONE_DIR / "py" / "pose_estimation" / "yolo26_pose" / "yolo26-s-pose_640x640"
POSE_MODEL = MODEL_DIR / "yolo26-s-pose_640x640.dxnn"
POSE_IMAGE = SAMPLE_IMG_DIR / "sample_people.jpg"

!cd "{POSE_DIR}" && "{DX_RUNTIME_PYTHON}" yolo26-s-pose_640x640_async_cpp_postprocess.py --model "{POSE_MODEL}" --image "{POSE_IMAGE}" --no-display --save --save-dir "{OUTPUT_DIR}"

show_latest_image(OUTPUT_DIR)

### 3.5 변형을 고르기 전에 측정하기

네 가지 변형 중 어느 것이 가장 빠른지는 시간이 어디에 쓰이는지에 달려 있습니다. 환경 변수 `DXAPP_PROGRESS=1`을 설정하면 모든 예제가 약 1초마다 `[PROGRESS]` 줄을 출력하고, 끝날 때 마지막 줄을 한 번 더 출력합니다. 각 줄에는 처리량(`fps`, `avg_fps`)과 각 단계의 프레임당 평균 시간이 들어 있습니다. 입력 읽기(`read_ms`), 전처리(`pre_ms`), NPU 추론(`infer_ms`), 후처리(`post_ms`)입니다.

다음 셀은 네 가지 YOLO26-S 변형을 모두 같은 588프레임 영상에 대해 헤드리스로 실행하고, 각 실행의 마지막 줄을 표로 정리합니다. 한 번의 실행은 다음과 같습니다.

```bash
DXAPP_PROGRESS=1 <dx-runtime>/venv-dx-runtime/bin/python yolo26-s_640x640_sync.py \
    --model <workspace>/res/models/yolo26-s_640x640.dxnn --video <dx_app>/assets/videos/boat.mp4 --no-display
```

이 셀은 약 40초가 걸립니다.

In [ ]:
VIDEO = DX_APP_DIR / "assets" / "videos" / "boat.mp4"
VARIANT_SCRIPTS = [
    "yolo26-s_640x640_sync.py",
    "yolo26-s_640x640_async.py",
    "yolo26-s_640x640_sync_cpp_postprocess.py",
    "yolo26-s_640x640_async_cpp_postprocess.py",
]

rows = []
for script in VARIANT_SCRIPTS:
    log = !cd "{EXAMPLE_DIR}" && DXAPP_PROGRESS=1 "{DX_RUNTIME_PYTHON}" {script} --model "{MODEL}" --video "{VIDEO}" --no-display 2>&1
    progress = [line for line in log if line.startswith("[PROGRESS]")]
    if not progress:
        print("\n".join(log[-20:]))
        raise RuntimeError(f"{script} printed no [PROGRESS] line; see its output above")
    rows.append((script, dict(re.findall(r"(\w+)=([\d.]+)", progress[-1]))))

print(f"{'variant':<44}{'avg_fps':>9}{'pre_ms':>8}{'infer_ms':>10}{'post_ms':>9}")
for script, m in rows:
    print(f"{script:<44}{m['avg_fps']:>9}{m['pre_ms']:>8}{m['infer_ms']:>10}{m['post_ms']:>9}")

표를 읽는 방법(수치는 호스트 CPU와 NPU 카드에 따라 다릅니다):

- **비동기와 동기 비교:** 비동기 변형은 여러 프레임을 동시에 NPU에서 처리하므로 훨씬 높은 프레임 속도에 도달합니다. DX-M1 카드 한 장에서 YOLO26-S는 비동기로 약 200 FPS, 동기로 약 50 FPS로 실행됩니다. 비동기 모드에서 `infer_ms`가 커지는 것은 NPU 시간만이 아니라 각 프레임을 제출해서 결과를 받기까지의 소요 시간을 측정하기 때문입니다.
- **C++와 Python 후처리 비교:** `cpp_postprocess` 변형은 `post_ms`를 줄이지만, YOLO26-S의 후처리는 1밀리초에 한참 못 미치므로 프레임 속도는 거의 바뀌지 않습니다. C++ 후처리는 세그멘테이션 마스크, 많은 앵커, 느린 호스트 CPU처럼 `post_ms`가 프레임 시간에서 큰 비중을 차지할 때 효과가 있습니다.
- **경험 법칙:** `*_async.py`로 시작해 `DXAPP_PROGRESS=1`로 측정하고, 후처리가 병목일 때만 `cpp_postprocess` 변형으로 바꾸세요.

### 3.6 직접 해 보기: Model Zoo의 아무 모델이나

예제들은 하나의 패턴을 따랐고, 이 패턴은 497개 변형 전부에 적용됩니다.

1. [Model Zoo](https://developer.deepx.ai/modelzoo/) 페이지에서 모델을 고르고, `./scripts/dx_tool.sh search <keyword>`로 그 예제를 찾습니다.
2. 그 자리에서 실행합니다. DX-APP 트리 안에서는 모델과 입력이 선택 사항입니다. `--model`이 없으면 예제는 `assets/models/<variant>.dxnn`을 불러오고 파일이 없으면 먼저 다운로드하며, 입력이 없으면 해당 작업의 기본 샘플 이미지를 사용합니다.
3. 이를 바탕으로 개발하려면 내보냅니다: `scripts/extract_model_package.sh <task>/<family>/<variant> --lang py --output-dir <dir>`.

다음 셀은 `config/model_registry.json`에서 변형을 찾아, 그 동기 Python 예제를 모델과 입력 없이 그 자리에서 실행합니다. 가장 짧은 형태의 실행입니다. 결과를 표시하고 내보내기 명령을 출력합니다. 다른 모델을 시도하려면 `VARIANT`를 바꾸세요. 예를 들어 `resnet50_224x224`(분류), `yolov12-n_640x640`(YOLOv12), `rtdetr-r18vd-6x_640x640`(RT-DETR)을 써 볼 수 있습니다. 아직 디스크에 없는 모델은 먼저 다운로드되며, 몇 초에서 1분 정도 걸립니다.

In [ ]:
import json

VARIANT = "scrfd-500m_640x640"   # change me: any variant (the .dxnn file stem) from config/model_registry.json

registry = json.loads((DX_APP_DIR / "config" / "model_registry.json").read_text())
entry = next((e for e in registry if VARIANT in (e["variant"], e["model_name"])), None)
if entry is None:
    raise KeyError(f"{VARIANT} is not in the registry. Search with: cd {DX_APP_DIR} && ./scripts/dx_tool.sh search {VARIANT}")

task, family, variant = entry["task"], entry["family"], entry["variant"]
SCRIPT = f"src/python_example/{task}/{family}/{variant}/{variant}_sync.py"
print(f"{variant}: task={task}, family={family}, model file={entry['dxnn_file']}\n")

# The shortest run: no --model and no input, so the example uses its default model and sample image.
!cd "{DX_APP_DIR}" && "{DX_RUNTIME_PYTHON}" {SCRIPT} --no-display --save --save-dir "{OUTPUT_DIR}"
show_latest_image(OUTPUT_DIR)

print("\nExport it as a standalone package:")
print(f'cd "{DX_APP_DIR}" && bash scripts/extract_model_package.sh {task}/{family}/{variant} --lang py --output-dir "{STANDALONE_DIR}"')

## 4. 멀티 모델 그래프: 코드 작성 없이 모델 연결하기

실제 애플리케이션은 프레임마다 둘 이상의 모델을 실행하는 경우가 많습니다. 검출 모델이 사람을 찾으면 두 번째 모델이 그 속성을 읽거나, 손바닥 검출 모델이 손을 찾으면 랜드마크 모델이 손가락 관절 위치를 잡는 식입니다. DX-APP v3.3.0은 이를 위해 **멀티 모델 그래프**를 추가했습니다. 파이프라인을 JSON 파일로 기술하면 실행 파일 하나가 그것을 실행합니다.

- **nodes**는 입력 소스와 모델이며, 각 모델은 변형 이름으로 지정합니다.
- **edges**는 노드를 연결합니다. `"roi"`가 있는 edge는 첫 번째 모델이 찾은 모든 박스를 잘라 내어 그 크롭을 다음 모델에 전달합니다.

수정할 C++ 코드도, 다시 빌드할 것도 없습니다. 다른 모델을 시도하려면 JSON 파일의 문자열 하나만 바꾸면 됩니다. 실행 파일은 `bin/multi_model_graph_sync`와 `bin/multi_model_graph_async`이고, 샘플 그래프는 `src/cpp_example/multi_model_graph/`에 있습니다.

| 옵션 | 효과 |
|---|---|
| `--graph <file>` | 그래프 실행 |
| `--check <file>` | NPU를 사용하지 않고 그래프와 모델 파일을 검증 |
| `--output <file>` | 합성된 결과 이미지 저장 |
| `--report <file>` | 모든 노드의 결과를 JSON으로 저장 |
| `--list-models` | 그래프에서 사용할 수 있는 모델 목록 출력(`--consumes roi`는 2단계 모델 목록 출력) |

> Python 모듈 `dx_graph`는 같은 그래프를 Python에서 실행합니다. 이 모듈은 `./build.sh --all`로만 빌드되므로, 이 튜토리얼에서는 실행 파일을 사용합니다.

### 4.1 그래프 실행 파일 빌드와 모델 다운로드

다음 셀은 DX-APP 빌드에 두 실행 파일이 아직 없으면 빌드하며, 약 1분이 걸립니다. 이어서 이 절의 그래프가 사용하는 추가 모델 세 개를 다운로드합니다. 손바닥 검출 모델, 손 랜드마크 모델, 사람 속성 모델입니다.

In [ ]:
%cd {DX_APP_DIR}
GRAPH_DIR = DX_APP_DIR / "src" / "cpp_example" / "multi_model_graph"
GRAPH_OUTPUT_DIR = WORKSPACE_DIR / "graph-outputs"
GRAPH_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not (DX_APP_DIR / "bin" / "multi_model_graph_async").is_file():
    !./build.sh --target multi_model_graph_sync multi_model_graph_async
!ls bin/multi_model_graph_*

!bash setup.sh --models mediapipe-hand-detector_192x192 mediapipe-hands-lite_224x224 deepmar_resnet50_224x224 --no-force

### 4.2 2단계 캐스케이드: 손바닥 검출 후 손 랜드마크

`hand_cascade.json`은 가장 작은 샘플 그래프입니다.

In [ ]:
!cat "{GRAPH_DIR}/hand_cascade.json"

소스 `cam`은 샘플 이미지입니다. `palm`은 전체 프레임에서 손을 찾습니다. `palm`에서 `landmark`로 가는 edge에는 `"roi"`가 있으므로, 각 손 박스가 15% 여백을 두고 잘려 랜드마크 모델에 전달됩니다.

`--check`는 NPU에서 무엇이든 실행하기 전에 그래프를 검증합니다. 각 모델을 확인하고, 그 모델이 무엇을 `produces`하고 `consumes`하는지 보여 주며, `.dxnn` 파일이 있는지 확인합니다. 이어서 `--graph`가 그래프를 실행하고, `--output`이 합성된 이미지를 저장하며, `--report`가 모든 노드의 결과를 기록합니다. 샘플 그래프는 `dx_app` 기준의 상대 경로를 사용하므로 그 디렉터리에서 실행하세요.

In [ ]:
%cd {DX_APP_DIR}

!./bin/multi_model_graph_async --check src/cpp_example/multi_model_graph/hand_cascade.json

!./bin/multi_model_graph_async --graph src/cpp_example/multi_model_graph/hand_cascade.json --output "{GRAPH_OUTPUT_DIR}/hand_cascade.jpg" --report "{GRAPH_OUTPUT_DIR}/hand_cascade.json"

display(Image(filename=str(GRAPH_OUTPUT_DIR / "hand_cascade.jpg"), width=480))

리포트에는 프레임마다 각 노드의 결과가 나열됩니다. 전체 프레임에서 실행된 모델은 `nodes` 아래에, 크롭에서 실행된 모델은 각 크롭이 잘려 나온 프레임 내 박스와 함께 `roi_nodes` 아래에 있습니다. 다음 셀은 첫 번째 프레임을 요약합니다.

In [ ]:
report = json.loads((GRAPH_OUTPUT_DIR / "hand_cascade.json").read_text())
frame = report["frames"][0]
for name, node in frame["nodes"].items():
    payload = node["payload"]
    print(f"{name:<10} full frame  {payload['shape']:<10} items={len(payload.get('items', []))}")
for name, crops in frame.get("roi_nodes", {}).items():
    for crop in crops:
        box = [round(v) for v in crop["origin"]["src_box"]]
        print(f"{name:<10} crop        {crop['payload']['shape']:<10} from box {box}")

### 4.3 JSON 편집으로 모델 교체하기

`cascade_od_attr.json`은 사람을 검출하고, 각 사람 크롭을 사람 속성 모델(성별, 연령대, 옷차림 등)인 `deepmar_resnet50_224x224`에 전달합니다. 이 그래프의 검출 모델은 `yolov8-n_640x640`인데, 이 튜토리얼에서는 다운로드하지 않았습니다. 다음 셀은 이를 다운로드하는 대신, 검출 모델을 3절의 모델인 `yolo26-s_640x640`으로 바꾼 그래프 사본을 workspace에 작성합니다. 사본이 `dx_app` 밖에 있으므로 셀은 소스 이미지 경로도 절대 경로로 바꿉니다. 그런 다음 편집한 그래프를 검증하고 실행합니다. 다시 빌드하는 것은 없습니다.

In [ ]:
graph = json.loads((GRAPH_DIR / "cascade_od_attr.json").read_text())
for node in graph["nodes"]:
    if node.get("type") == "source":
        node["uri"] = str(DX_APP_DIR / node["uri"])   # absolute path: the copy is outside dx_app
    if node["id"] == "od":
        node["model"] = "yolo26-s_640x640"           # the one-string model swap

MY_GRAPH = GRAPH_OUTPUT_DIR / "person_attributes_yolo26.json"
MY_GRAPH.write_text(json.dumps(graph, indent=2) + "\n")
!cat "{MY_GRAPH}"

!./bin/multi_model_graph_async --check "{MY_GRAPH}"
!./bin/multi_model_graph_async --graph "{MY_GRAPH}" --output "{GRAPH_OUTPUT_DIR}/person_attributes.jpg"

display(Image(filename=str(GRAPH_OUTPUT_DIR / "person_attributes.jpg"), width=640))

### 4.4 어떤 모델이 맞는지, 그리고 그 밖의 그래프

모든 모델이 모든 자리에 맞는 것은 아닙니다. 2단계 모델은 크롭을 받을 수 있어야 하며, `consumes` 열이 `roi` 또는 `either`여야 합니다. `--list-models --consumes roi`는 레지스트리에서 이런 모델을 걸러 내며, `file` 열은 `.dxnn`이 디스크에 있는지 보여 줍니다. 다음 셀은 처음 몇 줄과 전체 개수를 출력합니다.

이어서 `run_multi_model_demo.sh --list`가 샘플 그래프 목록을 출력합니다. 샘플 그래프는 캐스케이드(검출 모델이 속성, 재식별, 분류 모델에 결과를 전달), 팬아웃(같은 프레임에서 검출, 세그멘테이션, 포즈), 체인(검출 전에 초해상도 또는 저조도 개선), 그리고 필터링, 카운팅, 추적, 영역이나 선 확인을 수행하는 로직 노드를 다룹니다. `bash run_multi_model_demo.sh --check`는 NPU를 사용하지 않고 모든 샘플 그래프를 검증하며, 없는 모델마다 다운로드 명령을 출력합니다.

In [ ]:
%cd {DX_APP_DIR}

# Second-stage candidates: the first lines, then the number of models
!./bin/multi_model_graph_async --list-models --consumes roi | head -12
!./bin/multi_model_graph_async --list-models --consumes roi | tail -n +3 | wc -l

# The sample graphs and the demo scenarios
!bash run_multi_model_demo.sh --list

# Validate every sample graph (no NPU use; long output):
#!bash run_multi_model_demo.sh --check

## 5. 문제 해결

| 증상 | 표시되는 내용 | 원인 | 해결 |
|---|---|---|---|
| 이전 실행 파일 이름을 찾을 수 없음 | `./bin/yolov7_async: No such file or directory` | DX-APP v3.3.0은 실행 파일 이름을 변형 이름으로 지정함 | `./bin/yolov7_640x640_async` 같은 새 이름 사용. `ls bin`으로 목록 확인 |
| 실행 파일이 없음 | `./bin/<variant>_sync: No such file or directory` | 기본 빌드(`--minimal`)는 26개 데모 모델만 빌드함 | 터미널에서 `dx_app`의 `./build.sh --target <variant>` 또는 `./build.sh --all` 실행 |
| 불러오는 시점에 모델이 거부됨 | `.dxnn container v9 needs DX-RT >= 3.5.0, but this runtime is 3.4.1` 또는 `Model file format version 9 is not supported` | Model Zoo `2_5_0` 파일에는 DX-RT 3.5.0 이상이 필요함 | DX-Runtime 업데이트(튜토리얼 01의 3절). `dxrt-cli --version`으로 설치된 버전 확인 |
| `import dx_engine` 실패 | `dx_engine and DX-RT versions do not match` | `venv-dx-runtime`에 다른 DX-RT용으로 빌드된 `dx_engine`이 있음 | 둘 다 3.5.x가 되도록 DX-Runtime 재설치 |
| 모델을 불러올 수 없음 | `assets/models` 아래 경로와 함께 `Failed to load model` | 모델이 다운로드되지 않음 | 1절의 다운로드 셀을 해당 변형 이름으로 실행한 뒤 다시 실행 |
| 그래프 검증에서 모델 누락 보고 | `MISSING: 1 model file is not in .../assets/models` | 그래프가 다운로드되지 않은 모델을 지정함 | `--check`가 출력한 다운로드 명령 또는 `bash setup.sh --models <variant> --no-force` 실행 |
| 출력 창이 없음 | `cannot open display` 또는 창 없이 셀이 종료됨 | 그래픽 세션 없음(SSH 또는 헤드리스 호스트) | 데스크톱 세션에서 실행하거나, `--no-display --save`를 유지하고 저장된 결과 확인 |
| 카메라를 열 수 없음 | `Could not open camera 0` | `/dev/video0`이 없거나 사용자가 `video` 그룹에 속해 있지 않음 | `v4l2-ctl --list-devices`; `sudo usermod -aG video $USER` 실행 후 다시 로그인 |
| `run_demo.sh`가 메뉴 전에 멈춤 | `Failed to download demo models.` | 데모 모델 자동 다운로드 중 네트워크에 접속할 수 없음 | 네트워크에 연결하거나, 네트워크가 되는 상태에서 `./setup.sh --demo-models`를 한 번 실행 |
| 내보낸 Python 예제가 시작 시 멈춤 | `VariantConfigError: processor class '...' is exported by neither common.processors nor common.visualizers` | 기본 프루닝으로 내보내 `config.json`에 지정된 클래스가 빠짐(DX-APP v3.3.0) | `--no-prune`으로 다시 내보내기 |
| 내보낸 Python 예제가 `dx_engine`을 임포트하지 못함 | `ModuleNotFoundError: No module named 'dx_engine'` | DX-Runtime 환경 대신 Jupyter Python으로 실행됨 | 3절의 셀처럼 `<dx-runtime>/venv-dx-runtime/bin/python` 사용 |
| 내보낸 C++ 패키지가 구성(configure)되지 않음 | `Could not find a package configuration file provided by "dxrt"` | DX-RT가 설치되지 않았거나 사용자 지정 prefix 아래에 있음 | DX-Runtime 설치(튜토리얼 01의 3절) 또는 `cmake` 줄에 `-DCMAKE_PREFIX_PATH=<prefix>` 추가 |
| 내보낸 예제가 입력을 찾지 못함 | `Input file not found: sample/img/...` | 패키지에 샘플 이미지가 없음 | `dx_app/sample/img`의 파일을 `--image`(Python) 또는 `-i`(C++)로 전달 |

## 6. 요약

### 6.1 완료한 DX-APP 워크플로

```text
Load SDK paths
       │
       ▼
Download the selected models by variant name
       │
       ▼
Run C++ or Python examples headless and view the saved results
       │
       ├── Synchronous
       ├── Asynchronous
       └── PPU
       │
       ▼
Find an example with dx_tool.sh, export it, and measure its variants
       │
       ▼
Chain several models with a multi-model graph
```

### 6.2 학습 대시보드

| 영역 | 이제 할 수 있는 것 |
|---|---|
| SDK 경로 | `config.json`에서 `DX_APP_DIR`과 관련 경로 불러오기 |
| 이름 규칙 | `<task>/<family>/<variant>`의 변형 이름으로 예제, 실행 파일, 스크립트 찾기 |
| 모델 설정 | 실습에 필요한 모델과 영상만 변형 이름으로 다운로드 |
| 실행 방식 | 목표에 맞게 동기, 비동기, PPU 실행 중 선택 |
| 측정 | 하나를 고르기 전에 `DXAPP_PROGRESS=1`로 네 가지 Python 변형 비교 |
| 데모 런처 | 메뉴 또는 `--task`, `--mode`, `--input`으로 `run_demo.sh` 데모 시작 |
| 독립 실행형 코드 | `scripts/extract_model_package.sh`로 아무 모델의 예제나 내보내고, 모델 카드를 읽은 뒤 DX-APP 밖에서 빌드하고 실행 |
| Model Zoo 커버리지 | `config/model_registry.json` 또는 `dx_tool.sh search`로 모든 Model Zoo 모델의 DX-APP 예제 찾기 |
| 멀티 모델 그래프 | 여러 모델을 연결하는 그래프 JSON을 검증, 실행, 편집 |

### 6.3 완료 체크리스트

- [ ] DX-RT에서 DEEPX NPU가 인식되고 DX-RT가 3.5.0 이상인지 확인
- [ ] 필요한 DX-APP 모델과 영상 에셋 준비
- [ ] 엔드투엔드 추론 예제를 하나 이상 실행하고 저장된 결과 확인
- [ ] 사용 가능한 실행 방식 비교
- [ ] `dx_tool.sh search`로 대상 작업의 참조 구현 찾기
- [ ] `run_demo.sh` 열기
- [ ] 독립 실행형 Python 및 C++ 예제를 DX-APP 트리 밖에서 내보내고 빌드하고 실행
- [ ] 네 가지 Python 변형을 측정하고 모델에 맞는 하나 선택
- [ ] 직접 선택한 Model Zoo 모델 실행
- [ ] 멀티 모델 그래프를 실행하고 그 모델 중 하나 교체

> **다음:** 튜토리얼 03으로 이동해 학습된 모델을 조정하고, DXNN으로 컴파일하고, DX-APP 런타임 구성과 통합해 보세요.